In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 

# To read filenames and load images
from glob import glob
from tifffile import imread
from tqdm import tqdm

from cellpose import models, core
from cellpose.io import logger_setup 

use_GPU = core.use_gpu()
print(f'>>> GPU activated? {use_GPU}')
logger_setup();

>>> GPU activated? False
2024-07-26 13:01:30,404 [INFO] WRITING LOG OUTPUT TO /home/hpc/iwi5/iwi5171h/.cellpose/run.log
2024-07-26 13:01:30,406 [INFO] 
cellpose version: 	3.0.9.dev8+gc03958c 
platform:       	linux 
python version: 	3.8.19 
torch version:  	2.3.1


In [2]:
model_path = "/home/hpc/iwi5/iwi5171h/cellpose_src/dataset/calculated/closest_vertex_high_degrees_only_120/train/models/cellpose_300_epoch_cyto_1719234624.3807368"
model = models.CellposeModel(gpu=True, pretrained_model=model_path)

2024-07-26 13:01:30,433 [INFO] TORCH CUDA version not installed/working.
2024-07-26 13:01:30,434 [INFO] >>>> using CPU
2024-07-26 13:01:30,552 [INFO] >>>> loading model /home/hpc/iwi5/iwi5171h/cellpose_src/dataset/calculated/closest_vertex_high_degrees_only_120/train/models/cellpose_300_epoch_cyto_1719234624.3807368
2024-07-26 13:01:31,017 [INFO] >>>> model diam_mean =  30.000 (ROIs rescaled to this size during training)
2024-07-26 13:01:31,018 [INFO] >>>> model diam_labels =  13.872 (mean diameter of training ROIs)


In [3]:
model.diam_labels

13.872229

In [4]:
diameter = 12

In [5]:
# Input images and gt_masks

X = sorted(glob('/home/hpc/iwi5/iwi5171h/stardist_src/stardist/examples/2D/data/calculated_dataset/closest_vertex_high_degree/test/images/*.tif')) # sorted list of filepaths
X = list(map(imread, X)) # reead each file and collect in a new list

Y = sorted(glob('/home/hpc/iwi5/iwi5171h/stardist_src/stardist/examples/2D/data/calculated_dataset/closest_vertex_high_degree/test/masks/*.tif')) # sorted list of filepaths
Y = list(map(imread, Y))

n_channel = 1 if X[0].ndim == 2 else X[0].shape[-1]
axis_norm = (0,1) # normalize channel independently
# axis_norm = (0,1,2) # normalize channels jointly
if n_channel > 1:
    print("Normalizing image channels %s." % ('jointly' if axis_norm is None or 2 in axis_norm else 'independently'))


In [6]:
X[0].shape,X[0].ndim

((159, 120), 2)

In [7]:
assert len(Y) == len(X) , 'Number of images and gt_masks are not same'

In [8]:
len(X) # number of images

18

In [12]:
def calculate_iou(y_true, y_pred):
    intersection = np.logical_and(y_true, y_pred)
    union = np.logical_or(y_true, y_pred)
    iou_score = np.sum(intersection) / np.sum(union)
    return iou_score

def calculate_dice_coefficient(y_true, y_pred):
    intersection = np.sum(y_true * y_pred)
    return (2. * intersection) / (np.sum(y_true) + np.sum(y_pred))


In [13]:
# a function that takes images and gt_masks. 
# predicts pred_masks of all images.
# calculates the metric for each image and returns the average metric.
def calculate_metric(images, gt_masks):
    if len(images) != len(gt_masks):
        raise ValueError('The length of images and gt_masks should be the same.')
    
    # Prediction using model
    pred_masks = []
    for image in tqdm(images):
        mask, _, _ = model.eval(image,channels=[0,0],flow_threshold=0,cellprob_threshold=0.0)
        pred_masks.append(mask)
    
    # Calculate metrics
    total_iou = 0
    total_dice = 0
    num_masks = len(gt_masks)
    for gt_mask, pred_mask in zip(gt_masks,pred_masks):
        # convert masks to boolean arrays
        gt_mask_bool = gt_mask > 0
        pred_mask_bool = pred_mask > 0
        
        iou = calculate_iou(gt_mask_bool, pred_mask_bool)
        dice = calculate_dice_coefficient(gt_mask_bool, pred_mask_bool)

        total_iou += iou
        total_dice += dice 

    average_iou = total_iou / num_masks
    average_dice = total_dice / num_masks

    return average_iou, average_dice

    

In [14]:
iou_metric, dice_metric = calculate_metric(X, Y)
print(f"IoU Metric: {iou_metric}")
print(f"Dice Metric: {dice_metric}")

100%|██████████| 18/18 [00:20<00:00,  1.13s/it]

IoU Metric: 0.7725356508125114
Dice Metric: 0.8713961700698936
